# Team Ranking Model Process

The goal is to score complete VGC teams based on their expected relative tournament performance, eventually allowing the model to assist with team building and recommendations.

## Team Representation

Each row represents **one complete team** using sparse multi-hot encoding.

Features should preserve which Pokémon owns each attribute:

```text
pokemon=Incineroar
item:Incineroar=Safety_Goggles
ability:Incineroar=Intimidate
tera:Incineroar=Ghost
move:Incineroar=Fake_Out
move:Incineroar=Parting_Shot
```

This ensures that moving an item, move, or other attribute to a different Pokémon changes the team's representation. Team and move order do not matter.

A small number of team-level features can also be added, such as:

```text
protect_count = 3
fake_out_count = 2
pair:Incineroar+Rillaboom = 1
```

Missing values should be distinguished from attributes that are not applicable to a format.

## Performance Target

Instead of predicting only Top Cut, predict normalized tournament performance:

\[
performance = 1-\frac{placement-1}{event\_size-1}
\]

This produces values from `0` to `1`, where `1` represents first place and `0` represents last place.

This preserves more information than a binary Top Cut label and allows teams to be ranked by their predicted performance.

## Model Progression

Models and features will be tested incrementally:

1. Pokémon-only features with regularized regression.
2. Add Pokémon-specific items and abilities.
3. Add moves and Tera types.
4. Add selected team-level features and common Pokémon pairs.
5. Compare regression models with a Random Forest Regressor.
6. Test a dedicated ranking model such as `XGBRanker`.

For `XGBRanker`, each tournament becomes a ranking group, allowing the model to learn which teams should rank above others within the same event.

## Training and Evaluation

Training and testing should be split **chronologically by tournament**, keeping entire events together:

```text
Earlier Events → Training
Later Events   → Testing
```

`event_id` should only be used for grouping and splitting, never as a model feature.

Models should primarily be evaluated by how well their predicted rankings match actual tournament results using metrics such as:

- Spearman rank correlation
- NDCG@K
- Actual performance of the model's highest-ranked teams

## Overall Pipeline

```text
Tournament Database
        ↓
Team Multi-Hot Encoding
        ↓
Normalized Placement Target
        ↓
Event-Based Train/Test Split
        ↓
Regression / Random Forest
        ↓
Rank-Based Evaluation
        ↓
Add Detailed Build Features
        ↓
XGBoost Ranking Model
        ↓
Team Scoring / Recommendations
```

The final score represents **historical performance associated with a team build**, not a guaranteed placement or win probability. Player skill, matchups, and tournament variance also affect results.

In [1]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install sqlalchemy
%pip install scikit-learn
%pip install python-dotenv
%pip install psycopg2-binary

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sqlalchemy import create_engine, URL

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Get connection to DB

from pathlib import Path
from dotenv import load_dotenv

#Get env varibles from FetchData folder
env_path = Path("../../Fetch_Data/.env").resolve()
load_dotenv(env_path)

# Verify correct Path
print(env_path)
print(env_path.exists())

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5433")),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(url)

test_query = """
SELECT pokemon_name
FROM pokemon
"""

test_df = pd.read_sql(test_query, engine)
print(test_df.head())

/Users/brice/Documents/Projects/Pokemon_VGC_Analysis/Fetch_Data/.env
True
  pokemon_name
0    bulbasaur
1      ivysaur
2     venusaur
3   charmander
4   charmeleon


In [3]:
from sqlalchemy import text

all_teams_path = Path("../../analysis/all_teams_reg.sql").resolve()

with open(all_teams_path, "r") as file:
    query = text(file.read())
    
regulation = "I"

all_teams = pd.read_sql(
    query,
    engine,
    params={'regulation' : regulation}
)

team = all_teams[all_teams["team_id"] == 3145]
features = {}

for index, row in team.iterrows():
    pokemon = row["pokemon_name"]

    # Pokemon itself
    features[f"pokemon={pokemon}"] = 1

    # Item
    if pd.notna(row["item_name"]):
        features[f"item:{pokemon}={row['item_name']}"] = 1

    # Ability
    if pd.notna(row["ability_name"]):
        features[f"ability:{pokemon}={row['ability_name']}"] = 1

    # Tera type
    if pd.notna(row["tera_type"]):
        features[f"tera:{pokemon}={row['tera_type']}"] = 1

    # Move
    if pd.notna(row["move_name"]):
        features[f"move:{pokemon}={row['move_name']}"] = 1

print(features)


def create_team_features(team_df):

    features = {}

    for _, row in team_df.iterrows():

        pokemon = row["pokemon_name"]

        # Pokemon
        features[f"pokemon={pokemon}"] = 1

        # Item
        if pd.notna(row["item_name"]):
            features[f"item:{pokemon}={row['item_name']}"] = 1

        # Ability
        if pd.notna(row["ability_name"]):
            features[f"ability:{pokemon}={row['ability_name']}"] = 1

        # Tera
        if pd.notna(row["tera_type"]):
            features[f"tera:{pokemon}={row['tera_type']}"] = 1

        # Move
        if pd.notna(row["move_name"]):
            features[f"move:{pokemon}={row['move_name']}"] = 1

    return features


{'pokemon=ursaluna': 1, 'item:ursaluna=flame-orb': 1, 'ability:ursaluna=guts': 1, 'tera:ursaluna=ghost': 1, 'move:ursaluna=headlong-rush': 1, 'move:ursaluna=facade': 1, 'move:ursaluna=earthquake': 1, 'move:ursaluna=protect': 1, 'pokemon=urshifu-rapid-strike': 1, 'item:urshifu-rapid-strike=choice-band': 1, 'ability:urshifu-rapid-strike=unseen-fist': 1, 'tera:urshifu-rapid-strike=ghost': 1, 'move:urshifu-rapid-strike=surging-strikes': 1, 'move:urshifu-rapid-strike=u-turn': 1, 'move:urshifu-rapid-strike=close-combat': 1, 'move:urshifu-rapid-strike=aqua-jet': 1, 'pokemon=miraidon': 1, 'item:miraidon=assault-vest': 1, 'ability:miraidon=hadron-engine': 1, 'tera:miraidon=electric': 1, 'move:miraidon=draco-meteor': 1, 'move:miraidon=electro-drift': 1, 'move:miraidon=snarl': 1, 'move:miraidon=volt-switch': 1, 'pokemon=incineroar': 1, 'item:incineroar=rocky-helmet': 1, 'ability:incineroar=intimidate': 1, 'tera:incineroar=bug': 1, 'move:incineroar=fake-out': 1, 'move:incineroar=parting-shot': 1, 

In [ ]:
team_feature_dicts = []

team_ids = []

for team_id, team_df in all_teams.groupby("team_id"):

    team_feature_dicts.append(
        create_team_features(team_df)
    )

    team_ids.append(team_id)
    
print(len(team_feature_dicts))
print(all_teams["team_id"].nunique())

team_metadata = (
    all_teams[
        [
            "team_id",
            "event_id",
            "event_date",
            "regulation",
            "placement",
            "event_size"
        ]
    ]
    .drop_duplicates(subset="team_id")
    .set_index("team_id")
)





1017
1017


In [23]:
X = pd.DataFrame(
    team_feature_dicts,
    index=team_ids
)
X.index.name = "team_id"

X = X.fillna(0)
X = X.astype(int)

X = X.sort_index()
team_metadata = team_metadata.sort_index()

#print(X)
#print(X.shape)

X_pokemon = X.loc[
    :,
    X.columns.str.startswith("pokemon=")
]

X_pokemon_items = X.loc[
    :,
    X.columns.str.startswith(("pokemon=", "item:"))
]

X_pokemon_items_abilities = X.loc[
    :,
    X.columns.str.startswith(
        ("pokemon=", "item:", "ability:")
    )
]

X_pokemon_items_abilities_moves = X.loc[
    :,
    X.columns.str.startswith(
        ("pokemon=", "item:", "ability:", "move:")
    )
]

X_full = X

y = 1 - (
    (team_metadata["placement"] - 1)
    /
    (team_metadata["event_size"] - 1)
)

y = y.clip(0, 1).rename("performance")
print(y.head())
print(X.head())

print(X.index[:5])
print(team_metadata.index[:5])
#print(X.loc[3145][X.loc[3145] == 1])

team_id
494    1.000000
495    0.986842
496    0.973684
497    0.960526
498    0.947368
Name: performance, dtype: float64
         pokemon=sinistcha  item:sinistcha=eject-button  \
team_id                                                   
494                      1                            1   
495                      0                            0   
496                      0                            0   
497                      0                            0   
498                      0                            0   

         ability:sinistcha=hospitality  tera:sinistcha=normal  \
team_id                                                         
494                                  1                      1   
495                                  0                      0   
496                                  0                      0   
497                                  0                      0   
498                                  0                      0   

        

In [29]:
# Get each unique tournament and its date
events = (
    team_metadata[["event_id", "event_date"]]
    .drop_duplicates()
    .copy()
)

events["event_date"] = pd.to_datetime(events["event_date"])

# Oldest tournaments first
events = events.sort_values(
    by=["event_date", "event_id"]
).reset_index(drop=True)

# Split at 80% of tournaments
split_idx = int(len(events) * 0.8)

train_events = events.iloc[:split_idx]
test_events = events.iloc[split_idx:]

#print(f"Train events: {len(train_events)}")
#print(f"Test events: {len(events) - len(train_events)}")

#print(train_events)
#print(test_events)


# Get the IDs of training tournaments
train_ids = set(train_events["event_id"])

# Identify which teams belong to training tournaments
train_mask = team_metadata["event_id"].isin(train_ids).to_numpy()
test_mask = ~train_mask

# Split features
X_train = X.iloc[train_mask]
X_test = X.iloc[test_mask]

X_train_pokemon = X_pokemon.iloc[train_mask]
X_test_pokemon = X_pokemon.iloc[test_mask]

X_train_pokemon_items = X_pokemon_items.iloc[train_mask]
X_test_pokemon_items = X_pokemon_items.iloc[test_mask]

X_train_pokemon_items_abilities = X_pokemon_items_abilities.iloc[train_mask]
X_test_pokemon_items_abilities = X_pokemon_items_abilities.iloc[test_mask]

X_train_pokemon_items_abilities_moves = X_pokemon_items_abilities_moves.iloc[train_mask]
X_test_pokemon_items_abilities_moves = X_pokemon_items_abilities_moves.iloc[test_mask]

# Split target
y_train = y.iloc[train_mask]
y_test = y.iloc[test_mask]

print(X_train.head())
print(X_train_pokemon.head())

print(y_test.head())

         pokemon=sinistcha  item:sinistcha=eject-button  \
team_id                                                   
613                      0                            0   
614                      0                            0   
615                      0                            0   
616                      0                            0   
617                      1                            1   

         ability:sinistcha=hospitality  tera:sinistcha=normal  \
team_id                                                         
613                                  0                      0   
614                                  0                      0   
615                                  0                      0   
616                                  0                      0   
617                                  1                      1   

         move:sinistcha=shadow-ball  move:sinistcha=rage-powder  \
team_id                                                        

In [30]:
print(f"Training teams: {len(X_train)}")
print(f"Testing teams: {len(X_test)}")

print("Training date range:")
print(train_events["event_date"].min())
print(train_events["event_date"].max())

print("Testing date range:")
print(test_events["event_date"].min())
print(test_events["event_date"].max())

Training teams: 898
Testing teams: 119
Training date range:
2025-05-03 00:00:00
2026-05-16 00:00:00
Testing date range:
2026-05-16 00:00:00
2026-05-23 00:00:00


## Creating the Machine Learning DataFrames

The original SQL query returns the team data in **long format**, where each row represents one move belonging to one Pokémon on one team.

For example, the same Pokémon appears multiple times because it has multiple moves:

```text
team_id | pokemon_name | item_name | ability_name | tera_type | move_name
3145    | ursaluna     | flame-orb | guts         | ghost     | facade
3145    | ursaluna     | flame-orb | guts         | ghost     | protect
3145    | ursaluna     | flame-orb | guts         | ghost     | earthquake
...

# Models and Evaluation Metrics

## Models

The following models will be trained and compared to predict and rank Pokémon VGC tournament performance:

- **Ridge Regression:** A baseline model that estimates the individual contributions of Pokémon and their builds while reducing overfitting through regularization.
- **Gradient-Boosted Tree Regression:** A nonlinear model that can identify interactions between Pokémon, moves, items, abilities, and Tera types.
- **XGBRanker:** A learning-to-rank model that predicts the relative ordering of teams within individual tournaments.

## Evaluation Metrics

The models will be evaluated using the following metrics:

- **Spearman Rank Correlation (Primary):** Measures how closely the predicted ranking of teams matches their actual tournament placements. Ranges from -1 to 1, with 1 representing perfect agreement.
- **Kendall's Tau-b:** Measures agreement between predicted and actual rankings while accounting for tied placements.
- **NDCG@8 and NDCG@16:** Measures how effectively models place the strongest-performing teams near the top of their predicted rankings.
- **Mean Absolute Error (MAE):** Measures the average difference between predicted and actual normalized tournament performance for regression models. Lower values indicate better accuracy.

Ranking metrics will be calculated separately for each tournament and then aggregated across the test tournaments to compare model performance.

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error

model = RidgeCV(alphas=np.logspace(-3, 3, 13))
model.fit(X_train_pokemon, y_train)

y_pred = model.predict(X_test_pokemon)
y_pred = np.clip(y_pred, 0, 1)

print("Selected alpha:", model.alpha_)
print("Test MAE:", mean_absolute_error(y_test, y_pred))


Selected alpha: 100.0
Test MAE: 0.03968875667382507
Test Accuracy: 1.0
